In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path

In [3]:
def precision_at_q(y_true, y_hat, q=0.25):
    y_true = np.asarray(y_true)
    y_hat = np.asarray(y_hat)

    thr = np.quantile(y_true, q)
    true_pos = np.where(y_true <= thr)[0]
    k = len(true_pos)

    pred_topk = np.argsort(y_hat)[:k]

    return len(set(true_pos) & set(pred_topk)) / k

def ndcg_at_q(y_true, y_hat, q=0.25):
    "Normalized discounted cumulative gain"
    y_true = np.asarray(y_true)
    y_hat = np.asarray(y_hat)

    thr = np.quantile(y_true, q)
    true_pos = np.where(y_true <= thr)[0]
    k = len(true_pos)

    # relevance: higher is better
    rel = -y_true
    
    # predicted ranking
    order = np.argsort(y_hat)
    rel_pred = rel[order][:k]
    
    discounts = 1 / np.log2(np.arange(2, k + 2)) # rank weight
    dcg = np.sum((2 ** rel_pred - 1) * discounts)

    # ideal ranking
    ideal_order = np.argsort(y_true)
    rel_ideal = rel[ideal_order][:k]
    idcg = np.sum((2 ** rel_ideal - 1) * discounts)

    return dcg / idcg if idcg > 0 else 0.0

# Pancancer

In [2]:
root = Path("../../output/regression/cross_validation/")
cancertype = "pancancer"
experiment = "NBS_cells"

## NBS_cells | Fixed-drug | Fixed-cell

In [4]:
per_drug_pcc_cv = {}
per_cell_pcc_cv = {}

per_drug_precision_cv = {}
per_cell_precision_cv = {}

per_drug_ndcg_cv = {}
per_cell_ndcg_cv = {}

for fold_name in range(10):
    # train_path = root / type / experiment / f"fold_{n}" / "train_set.csv"
    test_path = root / cancertype / experiment / f"fold_{fold_name}" / "test_set.csv"
    
    y_hat = (
        pd.read_csv(f"CV/predictions_{cancertype}_{experiment}/model_{fold_name}.csv")
        .iloc[:,0]
    )

    # train_set = pd.read_csv(train_path)
    test_set = pd.read_csv(test_path)

    ## train stats
    # ic50_mean = train_set.LN_IC50.mean()
    # ic50_std = train_set.LN_IC50.std()

    ## test
    test_set["Y_TRUE"] = test_set["LN_IC50"]
    test_set["Y_HAT"] = y_hat

    # Computing metrics
    per_drug_pcc = (test_set.groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())

    per_drug_pcc_cv[fold_name] = [per_drug_pcc["PCC"].median()]

    per_cell_pcc = (test_set.groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())

    per_cell_pcc_cv[fold_name] = [per_cell_pcc["PCC"].median()]

    ##############################
    # PRECISION
    ##############################
    per_drug_pcc = (test_set.groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_drug_precision_cv[fold_name] = [per_drug_pcc["precision@q25"].median()]

    per_cell_pcc = (test_set.groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_cell_precision_cv[fold_name] = [per_cell_pcc["precision@q25"].median()]

    ##############################
    # NDCG
    ##############################
    per_drug_pcc = (test_set.groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_drug_ndcg_cv[fold_name] = [per_drug_pcc["ndcg@q25"].median()]

    per_cell_pcc = (test_set.groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_cell_ndcg_cv[fold_name] = [per_cell_pcc["ndcg@q25"].median()]

    


/tmp/ipykernel_21767/1030094531.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_21767/1030094531.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_21767/1030094531.py:53: Deprecation

In [5]:
per_drug_pcc_cv = pd.DataFrame(per_drug_pcc_cv)
per_drug_pcc_cv.index = ["HiDRA"]
per_drug_pcc_cv

,0,1,2,3,4,5,6,7,8,9
HiDRA,0.415448,0.387499,0.394137,0.423676,0.456916,0.37997,0.417523,0.448816,0.411269,0.306434


In [6]:
M = np.median(per_drug_pcc_cv)
sem = stats.sem(per_drug_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.41335842704929504, low=0.38316967226936516, high=0.4435471818292249


In [9]:
per_drug_pcc_cv.to_csv(
    "CV/predictions_pancancer_NBS_cells/fixed-drug_evaluation_df.csv"
)

In [7]:
per_cell_pcc_cv = pd.DataFrame(per_cell_pcc_cv)
per_cell_pcc_cv.index = ["CPV"]
per_cell_pcc_cv

,0,1,2,3,4,5,6,7,8,9
CPV,0.895133,0.889918,0.889804,0.889965,0.894828,0.890261,0.893631,0.889996,0.883816,0.880361


In [8]:
M = np.median(per_cell_pcc_cv)
sem = stats.sem(per_cell_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.8899809416318449, low=0.8866631881088882, high=0.8932986951548016


In [10]:
per_cell_pcc_cv.to_csv(
    "CV/predictions_pancancer_NBS_cells/fixed-cell_evaluation_df.csv"
)

In [7]:
per_drug_precision_cv = pd.DataFrame(per_drug_precision_cv)
per_drug_precision_cv.index = ["HiDRA"]
per_drug_precision_cv

,0,1,2,3,4,5,6,7,8,9
HiDRA,0.478261,0.478261,0.458333,0.478261,0.5,0.454545,0.458333,0.5,0.478261,0.416667


In [11]:
per_drug_precision_cv.to_csv(
    "CV/predictions_pancancer_NBS_cells/precision_fixed-drug_CV.csv"
)

In [8]:
per_drug_ndcg_cv = pd.DataFrame(per_drug_ndcg_cv)
per_drug_ndcg_cv.index = ["HiDRA"]
per_drug_ndcg_cv

,0,1,2,3,4,5,6,7,8,9
HiDRA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
per_drug_ndcg_cv.to_csv(
    "CV/predictions_pancancer_NBS_cells/ndcg_fixed-drug_CV.csv"
)

In [9]:
per_cell_precision_cv = pd.DataFrame(per_cell_precision_cv)
per_cell_precision_cv.index = ["HiDRA"]
per_cell_precision_cv

,0,1,2,3,4,5,6,7,8,9
HiDRA,0.803002,0.793663,0.788889,0.797619,0.80198,0.786885,0.8,0.787234,0.794118,0.788462


In [13]:
per_cell_precision_cv.to_csv(
    "CV/predictions_pancancer_NBS_cells/precision_fixed-cell_CV.csv"
)

In [10]:
per_cell_ndcg_cv = pd.DataFrame(per_cell_ndcg_cv)
per_cell_ndcg_cv.index = ["HiDRA"]
per_cell_ndcg_cv

,0,1,2,3,4,5,6,7,8,9
HiDRA,0.814951,0.838799,0.810563,0.813052,0.82073,0.821705,0.812117,0.810207,0.813499,0.800994


In [14]:
per_cell_ndcg_cv.to_csv(
    "CV/predictions_pancancer_NBS_cells/ndcg_fixed-cell_CV.csv"
)

## NBS_cells | Fixed-pathway | Fixed-TCGA

In [3]:
per_path_pcc_cv = {}
per_tcga_pcc_cv = {}

for fold_name in range(10):
    # train_path = root / cancertype / experiment / f"fold_{n}" / "train_set.csv"
    test_path = root / cancertype / experiment / f"fold_{fold_name}" / "test_set.csv"
    
    y_hat = (
        pd.read_csv(f"CV/predictions_{cancertype}_{experiment}/model_{fold_name}.csv")
        .iloc[:,0]
    )

    # train_set = pd.read_csv(train_path)
    test_set = pd.read_csv(test_path)

    ## train stats
    # ic50_mean = train_set.LN_IC50.mean()
    # ic50_std = train_set.LN_IC50.std()

    ## test
    test_set["Y_TRUE"] = test_set["LN_IC50"]
    test_set["Y_HAT"] = y_hat

    pathway = (test_set[["DRUG_NAME","PATHWAY_NAME"]]
             .drop_duplicates())
    
    tcga_desc = (test_set[["CELL_LINE_NAME","TCGA_DESC"]]
             .drop_duplicates())

    # Computing metrics
    # Computing metrics
    per_path_pcc = (test_set
                    .groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    per_path_pcc = (per_path_pcc
                    .merge(pathway, on="DRUG_NAME", how="left")
                    .dropna(subset=["PATHWAY_NAME"])
                    .groupby("PATHWAY_NAME")["PCC"]
                    .median()
                    )

    per_path_pcc_cv[fold_name] = per_path_pcc

    per_tcga_pcc = (test_set
                    .groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    
    per_tcga_pcc = (per_tcga_pcc.
                    merge(tcga_desc, on="CELL_LINE_NAME", how="left").
                    dropna(subset="TCGA_DESC")
                    .groupby("TCGA_DESC")["PCC"]
                    .median()
                    )

    per_tcga_pcc_cv[fold_name] = per_tcga_pcc

    


/tmp/ipykernel_76367/3567348838.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_76367/3567348838.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_76367/3567348838.py:36: Deprecation

In [5]:
exclude = ["Unclassified", "Other", "Other, kinases", "Chromatin other"]

In [7]:
path = (pd.DataFrame(per_path_pcc_cv)
 .groupby(level=0)
 .median()
 .drop(exclude)
#  .median(axis=1)
 .rename_axis("")
 .T
 )

path["fold"] = path.index

path = (path
        .melt(id_vars="fold", var_name="PATHWAY_NAME", value_name="MCorrelation")
        )

path = path.assign(model="HiDRA")
path.to_csv("CV/predictions_pancancer_NBS_cells/fixed-drug_PATHWAY_CV.csv")
path

,fold,PATHWAY_NAME,MCorrelation,model
0,0,ABL signaling,0.448990,HiDRA
1,1,ABL signaling,0.255811,HiDRA
2,2,ABL signaling,0.288011,HiDRA
3,3,ABL signaling,0.458980,HiDRA
4,4,ABL signaling,0.613359,HiDRA
...,...,...,...,...
195,5,p53 pathway,0.479538,HiDRA
196,6,p53 pathway,0.429155,HiDRA
197,7,p53 pathway,0.487495,HiDRA
198,8,p53 pathway,0.471839,HiDRA


In [9]:
tcga = (pd.DataFrame(per_tcga_pcc_cv)
        .dropna()
        # .median(axis=1)
        .drop(["UNCLASSIFIED"])
        .rename_axis("")
 )

tcga = pd.DataFrame(tcga).T

tcga["fold"] = tcga.index

tcga = (tcga
        .melt(id_vars = "fold", var_name="TCGA_DESC", value_name="MCorrelation")
        )


tcga = tcga.assign(model="HiDRA")
tcga.to_csv("CV/predictions_pancancer_NBS_cells/fixed-cell_TCGA_CV.csv")
tcga

,fold,TCGA_DESC,MCorrelation,model
0,0,ALL,0.918405,HiDRA
1,1,ALL,0.893903,HiDRA
2,2,ALL,0.868881,HiDRA
3,3,ALL,0.858894,HiDRA
4,4,ALL,0.915739,HiDRA
...,...,...,...,...
115,5,SKCM,0.906199,HiDRA
116,6,SKCM,0.913553,HiDRA
117,7,SKCM,0.891952,HiDRA
118,8,SKCM,0.866406,HiDRA
